In [3]:
import os
import fitz as fitz # PyMuPDF
import nltk
from transformers import AutoTokenizer

nltk.download("punkt")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


/home/hazem/dev/Assist/glob_env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[nltk_data] Downloading package punkt to /home/hazem/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
# Testing fitz
import gc

print(fitz.__doc__)
# Open a PDF file and print its metadata
a = fitz.open("data/Structure.pdf")
print(a.metadata)
del a
gc.collect()    

PyMuPDF 1.24.11: Python bindings for the MuPDF 1.24.10 library (rebased implementation).
Python 3.8 running on linux (64-bit).

{'format': 'PDF 1.4', 'title': 'Structure', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'Skia/PDF m138 Google Docs Renderer', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None}


0

In [6]:
def extract_text_from_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def extract_text_from_txt(txt_path: str) -> str:
    with open(txt_path, "r", encoding="utf-8") as f:
        return f.read()

In [7]:
def tokenize_and_chunk(text: str, max_tokens=500, overlap=50):
    # Simple whitespace cleanup
    text = " ".join(text.split())

    tokens = tokenizer.tokenize(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.convert_tokens_to_string(chunk_tokens)
        chunks.append(chunk_text)
        if end == len(tokens):
            break
        start += max_tokens - overlap
    return chunks

In [9]:
def load_documents_with_metadata(root_folder: str):
    documents = []
    for dirpath, _, files in os.walk(root_folder):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            file_path = os.path.join(dirpath, file)
            rel_path = os.path.relpath(file_path, root_folder)

            try:
                if ext == ".pdf":
                    text = extract_text_from_pdf(file_path)
                elif ext in [".txt", ".md"]:
                    text = extract_text_from_txt(file_path)
                else:
                    print(f"Skipping unsupported file type: {file}")
                    continue
            except Exception as e:
                print(f"Failed to extract {file}: {e}")
                continue

            chunks = tokenize_and_chunk(text)
            documents.append({"file_path": rel_path, "chunks": chunks})
            print(f"Processed {rel_path}, {len(chunks)} chunks")

    return documents

In [10]:
data_folder = "data"
documents = load_documents_with_metadata(data_folder)
print(f"Loaded {len(documents)} documents in total.")

Token indices sequence length is longer than the specified maximum sequence length for this model (10675 > 512). Running this sequence through the model will result in indexing errors


Processed Structure.pdf, 1 chunks
Skipping unsupported file type: scientific-writing.zip
Processed Data Science Seminar/Lecture Notes/Derntl-Research-Paper-Writing.pdf, 24 chunks
Processed Data Science Seminar/Lecture Notes/Zobel-Chapter3-Reading-And-Reviewing.pdf, 0 chunks
Processed Data Science Seminar/Lecture Notes/Giving_a_talk.pdf, 4 chunks
Processed Data Science Seminar/Lecture Notes/Zobel-Questions-Hypothesis-Evidence.pdf, 22 chunks
Processed Data Science Seminar/Lecture Notes/Zobel-Chapter16_17-Presentation-and-Ethics.pdf, 0 chunks
Processed Data Science Seminar/Lecture Notes/part-1-en-publication-process.pdf, 9 chunks
Skipping unsupported file type: scientific-writing-reader.slides.html
Skipping unsupported file type: scientific-writing-overview.slides.html
Skipping unsupported file type: scientific-writing-paper-structure.slides.html
Skipping unsupported file type: reader_mindmap.png
Processed Data Science Seminar/Lecture Notes/scientific-writing/media/img/reader_mindmap.pdf,

In [11]:
import json

def save_chunks_to_disk(documents, save_folder="preprocessed"):
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)

    for doc in documents:
        filename = os.path.basename(doc["file_path"]) + ".json"
        save_path = os.path.join(save_folder, filename)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(doc, f, ensure_ascii=False, indent=2)
        print(f"Saved chunks for {doc['file_path']} to {save_path}")

# Usage
save_chunks_to_disk(documents)

Saved chunks for Structure.pdf to preprocessed/Structure.pdf.json
Saved chunks for Data Science Seminar/Lecture Notes/Derntl-Research-Paper-Writing.pdf to preprocessed/Derntl-Research-Paper-Writing.pdf.json
Saved chunks for Data Science Seminar/Lecture Notes/Zobel-Chapter3-Reading-And-Reviewing.pdf to preprocessed/Zobel-Chapter3-Reading-And-Reviewing.pdf.json
Saved chunks for Data Science Seminar/Lecture Notes/Giving_a_talk.pdf to preprocessed/Giving_a_talk.pdf.json
Saved chunks for Data Science Seminar/Lecture Notes/Zobel-Questions-Hypothesis-Evidence.pdf to preprocessed/Zobel-Questions-Hypothesis-Evidence.pdf.json
Saved chunks for Data Science Seminar/Lecture Notes/Zobel-Chapter16_17-Presentation-and-Ethics.pdf to preprocessed/Zobel-Chapter16_17-Presentation-and-Ethics.pdf.json
Saved chunks for Data Science Seminar/Lecture Notes/part-1-en-publication-process.pdf to preprocessed/part-1-en-publication-process.pdf.json
Saved chunks for Data Science Seminar/Lecture Notes/scientific-writi